# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [2]:
# 답안

import torch
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self):
        # 데이터 30개와 각 데이터의 입력 특성 4개를 생성한다.
        self.data = torch.randn(30, 4)

    def __len__(self):
        # 전체 데이터의 개수인 30을 반환한다.
        return len(self.data)

    def __getitem__(self, idx):
        # 주어진 인덱스에 해당하는 데이터를 반환한다.
        return self.data[idx]


# Dataset 객체 생성
dataset = MyDataset()

# 결과 확인
print("데이터 개수:", len(dataset))
print("첫 번째 데이터:", dataset[0])
print("데이터 형태:", dataset.data.shape)


데이터 개수: 30
첫 번째 데이터: tensor([-1.1258, -1.1524, -0.2506, -0.4339])
데이터 형태: torch.Size([30, 4])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [3]:
# 답안

import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self):
        # 데이터 30개와 각 데이터의 입력 특성 4개를 생성한다.
        self.data = torch.randn(30, 4)
        # 각 데이터의 정답값 30개를 생성한다.
        self.targets = torch.randn(30, 1)

    def __len__(self):
        # 전체 데이터 개수를 반환한다.
        return len(self.data)

    def __getitem__(self, idx):
        # 해당 인덱스의 입력 데이터와 정답값을 반환한다.
        return self.data[idx], self.targets[idx]


# Dataset 생성
dataset = MyDataset()

# 데이터를 7개씩 묶는 DataLoader 생성
dataloader = DataLoader(dataset, batch_size=7, shuffle=False)

# 전체 batch 개수 출력
print("Batch 개수:", len(dataloader))

# 각 batch의 입력 shape과 정답 shape 출력
for i, (x, y) in enumerate(dataloader):
    print(f"Batch {i+1}: 입력 shape = {x.shape}, 정답 shape = {y.shape}")


Batch 개수: 5
Batch 1: 입력 shape = torch.Size([7, 4]), 정답 shape = torch.Size([7, 1])
Batch 2: 입력 shape = torch.Size([7, 4]), 정답 shape = torch.Size([7, 1])
Batch 3: 입력 shape = torch.Size([7, 4]), 정답 shape = torch.Size([7, 1])
Batch 4: 입력 shape = torch.Size([7, 4]), 정답 shape = torch.Size([7, 1])
Batch 5: 입력 shape = torch.Size([2, 4]), 정답 shape = torch.Size([2, 1])


## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


답:입력 데이터의 batch shape은 (batch_size, 특성 수)를 의미합니다. 첫 번째 숫자는 한 번에 처리하는 데이터의 개수이고, 두 번째 숫자는 각 데이터가 가지고 있는 특성의 개수입니다.
예를 들어 torch.Size([7, 4])는 데이터 7개가 들어 있고, 각 데이터마다 특성이 4개 있다는 뜻입니다. 마지막 batch는 전체 데이터 30개 중 28개를 사용하고 2개가 남았기 때문에 torch.Size([2, 4])로 나타납니다.
정답 shape인 torch.Size([7, 1])은 데이터 7개에 대한 정답값이 각각 1개씩 있다는 의미입니다.

## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [4]:
# 답안

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self):
        # 입력 데이터 30개와 각 데이터의 특성 4개를 생성한다.
        self.data = torch.randn(30, 4)
        # 각 입력 데이터에 대한 정답값을 생성한다.
        self.targets = torch.randn(30, 1)

    def __len__(self):
        # 전체 데이터 개수를 반환한다.
        return len(self.data)

    def __getitem__(self, idx):
        # 해당 인덱스의 입력 데이터와 정답값을 반환한다.
        return self.data[idx], self.targets[idx]


# Dataset과 DataLoader 생성
dataset = MyDataset()
dataloader = DataLoader(dataset, batch_size=7, shuffle=True)

# 입력 특성 4개를 받아 정답 1개를 예측하는 선형 모델 생성
model = nn.Linear(4, 1)

# 손실 함수와 최적화 함수 설정
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 모델 학습
for epoch in range(20):
    total_loss = 0

    for x, y in dataloader:
        # 1. 예측: 입력 데이터를 모델에 넣어 예측값을 구한다.
        prediction = model(x)

        # 2. 손실: 예측값과 실제 정답의 차이를 계산한다.
        loss = criterion(prediction, y)

        # 3. 초기화: 이전 단계에서 계산된 기울기를 초기화한다.
        optimizer.zero_grad()

        # 4. 역전파: 손실을 바탕으로 기울기를 계산한다.
        loss.backward()

        # 5. 수정: 계산된 기울기를 이용해 모델의 가중치를 수정한다.
        optimizer.step()

        total_loss += loss.item()

    # epoch마다 평균 손실 출력
    average_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}, 평균 손실: {average_loss:.4f}")


Epoch 1, 평균 손실: 0.8647
Epoch 2, 평균 손실: 0.7537
Epoch 3, 평균 손실: 0.8685
Epoch 4, 평균 손실: 0.6707
Epoch 5, 평균 손실: 0.5899
Epoch 6, 평균 손실: 0.5889
Epoch 7, 평균 손실: 0.5890
Epoch 8, 평균 손실: 0.5774
Epoch 9, 평균 손실: 0.5177
Epoch 10, 평균 손실: 0.5204
Epoch 11, 평균 손실: 0.5973
Epoch 12, 평균 손실: 0.5412
Epoch 13, 평균 손실: 0.6203
Epoch 14, 평균 손실: 0.5975
Epoch 15, 평균 손실: 0.5739
Epoch 16, 평균 손실: 0.5205
Epoch 17, 평균 손실: 0.4856
Epoch 18, 평균 손실: 0.6210
Epoch 19, 평균 손실: 0.4940
Epoch 20, 평균 손실: 0.4784
